In [ ]:
# ==============================================================
# 🧠 CONVERT ADNI + CCNA LANGUAGE PKL RESULTS → compare_data.json
# ==============================================================
import pickle, json, numpy as np

# --- Load both pickle result files ---
with open("adni_language_results.pkl", "rb") as f:
    Adni = pickle.load(f)

with open("ccna_language_results.pkl", "rb") as f:
    ccna = pickle.load(f)

# --- Helper to make JSON-serializable ---
def make_json_serializable(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [make_json_serializable(v) for v in obj]
    elif isinstance(obj, (np.float32, np.float64, np.int32, np.int64)):
        return float(obj)
    else:
        return obj

# --- Prepare final comparison data ---
data_out = {
    "ADNI": {
        "y_pred": make_json_serializable(adni.get("y_pred")),
        "p_pred": make_json_serializable(adni.get("p_pred")),
        "cm": make_json_serializable(adni.get("cm")),
        "shap_importance": make_json_serializable(adni.get("shap_importance")),
        "silhouette_score": make_json_serializable(adni.get("silhouette_score"))
    },
    "CCNA": {
        "y_pred": make_json_serializable(ccna.get("y_pred")),
        "p_pred": make_json_serializable(ccna.get("p_pred")),
        "cm": make_json_serializable(ccna.get("cm")),
        "shap_importance": make_json_serializable(ccna.get("shap_importance")),
        "silhouette_score": make_json_serializable(ccna.get("silhouette_score"))
    }
}

# --- Save as JSON file ---
with open("compare_data.json", "w") as f:
    json.dump(data_out, f, indent=2)

print("\n💾 Saved: compare_data.json — ready for cross-cohort metric computation.")



💾 Saved: compare_data.json — ready for cross-cohort metric computation.


In [ ]:
# ==============================================================
# 🧩 FULL CROSS-COHORT SIMILARITY (ADNI vs CCNA LANGUAGE)
# ==============================================================
import json, numpy as np, pandas as pd
from scipy.spatial.distance import jensenshannon
from scipy.stats import pearsonr, kendalltau
from math import sqrt

# --------------------------------------------------------------
# Load compare_data.json (from your saved ADNI + CCNA PKLs)
# --------------------------------------------------------------
with open("compare_data.json", "r") as f:
    d = json.load(f)

# If your JSON keys are capitalized, adjust here:
if "ADNI" in d and "CCNA" in d:
    A, C = d["ADNI"], d["CCNA"]
else:
    A, C = d["adni"], d["ccna"]

# --------------------------------------------------------------
# Flatten probability arrays and align lengths
# --------------------------------------------------------------
pA = np.array(A["p_pred"], dtype=float).ravel()
pC = np.array(C["p_pred"], dtype=float).ravel()
n = min(len(pA), len(pC))
pA, pC = pA[:n], pC[:n]

# --------------------------------------------------------------
# 1️⃣ Jensen–Shannon Similarity
# --------------------------------------------------------------
js_vals = []
for pa, pc in zip(pA, pC):
    pa = float(np.clip(pa, 1e-9, 1-1e-9))
    pc = float(np.clip(pc, 1e-9, 1-1e-9))
    P, Q = np.array([pa, 1-pa]), np.array([pc, 1-pc])
    js_vals.append(jensenshannon(P, Q, base=2)**2)
JS = 1 - np.mean(js_vals)

# --------------------------------------------------------------
# 2️⃣ Bhattacharyya Coefficient
# --------------------------------------------------------------
BC = np.mean([
    np.sum(np.sqrt(np.array([pa, 1-pa]) * np.array([pc, 1-pc])))
    for pa, pc in zip(pA, pC)
])

# --------------------------------------------------------------
# 3️⃣ Pearson Similarity (mapped to [0, 1])
# --------------------------------------------------------------
pearson_corr, _ = pearsonr(pA, pC)
Pearson_sim = (pearson_corr + 1) / 2

# --------------------------------------------------------------
# 4️⃣ Hellinger Similarity (1 − H)
# --------------------------------------------------------------
Hellinger_dist = np.mean([
    sqrt(0.5 * np.sum(
        (np.sqrt(np.array([pa,1-pa])) -
         np.sqrt(np.array([pc,1-pc])))**2))
    for pa, pc in zip(pA, pC)
])
Hellinger_sim = 1 - Hellinger_dist

# --------------------------------------------------------------
# 5️⃣ Kendall’s τ (rank correlation)
# --------------------------------------------------------------
τ, _ = kendalltau(pA, pC)
τ = 0 if np.isnan(τ) else τ

# --------------------------------------------------------------
# 6️⃣ Composite Scores
# --------------------------------------------------------------
R_prob = np.mean([JS, BC, Pearson_sim, Hellinger_sim])
R_all  = np.mean([JS, BC, Pearson_sim, Hellinger_sim, τ])

# --------------------------------------------------------------
# 7️⃣ Bootstrap 95 % CI for R_prob
# --------------------------------------------------------------
boot = []
for _ in range(500):
    idx = np.random.choice(range(n), size=n, replace=True)
    pbA, pbC = pA[idx], pC[idx]
    js_vals = [jensenshannon(np.array([pa,1-pa]),
                              np.array([pc,1-pc]), base=2)**2
               for pa, pc in zip(pbA, pbC)]
    jsb = 1 - np.mean(js_vals)
    bcb = np.mean([np.sum(np.sqrt(np.array([pa,1-pa]) *
                                  np.array([pc,1-pc])))
                   for pa, pc in zip(pbA, pbC)])
    pearb, _ = pearsonr(pbA, pbC)
    pearb = (pearb + 1) / 2
    hellb = 1 - np.mean([
        sqrt(0.5 * np.sum(
            (np.sqrt(np.array([pa,1-pa])) -
             np.sqrt(np.array([pc,1-pc])))**2))
        for pa, pc in zip(pbA, pbC)
    ])
    boot.append(np.mean([jsb, bcb, pearb, hellb]))

boot = np.array(boot)
boot_mean = np.mean(boot)
boot_low, boot_high = np.percentile(boot, [2.5, 97.5])

# --------------------------------------------------------------
# 8️⃣ Create results table
# --------------------------------------------------------------
out = pd.DataFrame({
    "Metric": [
        "JS Similarity (prob.)",
        "Bhattacharyya Coefficient",
        "Pearson Similarity (mapped to [0,1])",
        "Hellinger Similarity (1 − H)",
        "Kendall’s τ (risk ranks)",
        "Composite R_prob (JS, BC, Pearson, Hellinger)",
        "Composite R_all (+ τ)",
        "Bootstrap R_prob mean",
        "Bootstrap R_prob 95 % CI low",
        "Bootstrap R_prob 95 % CI high"
    ],
    "Value": [
        JS, BC, Pearson_sim, Hellinger_sim, τ,
        R_prob, R_all, boot_mean, boot_low, boot_high
    ]
})

# --------------------------------------------------------------
# 9️⃣ Display + Save to compute_data.json
# --------------------------------------------------------------
print(out.to_string(index=False))

results_json = {
    "Cross_Cohort_Similarity": {
        "JS_Similarity": float(JS),
        "Bhattacharyya_Coefficient": float(BC),
        "Pearson_Similarity": float(Pearson_sim),
        "Hellinger_Similarity": float(Hellinger_sim),
        "Kendall_Tau": float(τ),
        "Composite_R_prob": float(R_prob),
        "Composite_R_all": float(R_all),
        "Bootstrap_R_prob_mean": float(boot_mean),
        "Bootstrap_R_prob_CI_low": float(boot_low),
        "Bootstrap_R_prob_CI_high": float(boot_high)
    }
}

with open("compute_data.json", "w") as f:
    json.dump(results_json, f, indent=4)

print("\n💾 Saved: compute_data.json — cross-cohort metrics ready for fusion analysis.")


                                       Metric     Value
                        JS Similarity (prob.)  0.677034
                    Bhattacharyya Coefficient  0.689259
         Pearson Similarity (mapped to [0,1])  0.559240
                 Hellinger Similarity (1 − H)  0.666851
                     Kendall’s τ (risk ranks) -0.131240
Composite R_prob (JS, BC, Pearson, Hellinger)  0.648096
                        Composite R_all (+ τ)  0.492228
                        Bootstrap R_prob mean  0.648418
                 Bootstrap R_prob 95 % CI low  0.636492
                Bootstrap R_prob 95 % CI high  0.660998

💾 Saved: compute_data.json — cross-cohort metrics ready for fusion analysis.
